In [1]:
!pip install -Uq "trl[peft]" trackio bitsandbytes liger-kernel wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 31.1 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.6/230.6 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 78.7 MB/s eta 0:00:00:00:010:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 95.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 97.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 21.4 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import login

login(token=hf_token)

In [ ]:
import wandb

wandb.login(wanbd_token)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: leviettin to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
import os
os.environ["WANDB_PROJECT"] = "DPOTraining"  # name your W&B project

In [6]:
model_id, output_dir = "meta-llama/Llama-3.2-3B-Instruct", "Llama-3.2-3B-Instruct-DPO"
# model_id, output_dir = "Qwen/Qwen2.5-VL-3B-Instruct", "Qwen2.5-VL-3B-Instruct-DPO"

In [7]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",                   # Change to Flash Attention if GPU has support
    dtype=torch.float16,                          # Change to bfloat16 if GPU has support
    device_map="auto",
    # use_cache=True,                               # Whether to cache attention outputs to speed up inference
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,                        # Load the model in 4-bit precision to save memory
        bnb_4bit_compute_dtype=torch.float16,     # Data type used for internal computations in quantization
        bnb_4bit_use_double_quant=True,           # Use double quantization to improve accuracy
        bnb_4bit_quant_type="nf4"                 # Type of quantization. "nf4" is recommended for recent LLMs
    )
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

2026-01-09 02:25:59.615967: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767925560.036705      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767925560.167021      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767925561.144965      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767925561.145009      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767925561.145012      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [8]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig, AutoTokenizer, AutoProcessor
from peft import PeftModel


base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_id,
    attn_implementation="sdpa",                   # Change to Flash Attention if GPU has support
    dtype=torch.float16,                          # Change to bfloat16 if GPU has support
    device_map="auto",
    # use_cache=True,                               # Whether to cache attention outputs to speed up inference
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,                        # Load the model in 4-bit precision to save memory
        bnb_4bit_compute_dtype=torch.float16,     # Data type used for internal computations in quantization
        bnb_4bit_use_double_quant=True,           # Use double quantization to improve accuracy
        bnb_4bit_quant_type="nf4"                 # Type of quantization. "nf4" is recommended for recent LLMs
    )
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
processor = AutoProcessor.from_pretrained(model_id)

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.53G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.98G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
adapter_model = f"JellyFush/Llama-3.2-3B-Instruct"
# adapter_model = f"JellyFush/Qwen2.5-VL-3B-Instruct"
model = PeftModel.from_pretrained(
    base_model,
    adapter_model,
    adapter_name="policy",
    is_trainable=True
)

# ---- Load REFERENCE adapter (frozen) ----
model.load_adapter(
    adapter_model,
    adapter_name="ref",
    is_trainable=False
)

# Set the active adapter to policy for training
model.set_adapter("policy")

# Enable input gradients (required for gradient checkpointing with quantized models)
model.enable_input_require_grads()

# Enable gradient checkpointing with use_reentrant=False (required for quantized models)
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

# Verify trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} || All params: {all_params:,} || Trainable%: {100 * trainable_params / all_params:.2f}%")

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

Trainable params: 24,313,856 || All params: 1,852,091,392 || Trainable%: 1.31%


In [9]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [10]:
# Download file from Google Drive
import os
import gdown

# https://drive.google.com/file/d/1ho5GuoXy0laRI450RFQ8WR9x4flz1jy9/view?usp=sharing - qwen
# https://drive.google.com/file/d/1yv1BO7y1VdPBG3hm8A09twAFpVSedxp-/view?usp=sharing - llama
# Google Drive file ID from the sharing link, then change the ID
file_id = "1yv1BO7y1VdPBG3hm8A09twAFpVSedxp-"
# URL for downloading
url = f"https://drive.google.com/uc?id={file_id}"

output_path = "/content/full_dpo_dataset_llama.json"

# Download the file
gdown.download(url, output_path, quiet=False)

print(f"Downloaded to {output_path}")


Downloading...
From: https://drive.google.com/uc?id=1yv1BO7y1VdPBG3hm8A09twAFpVSedxp-
To: /content/full_dpo_dataset_llama.json
100%|██████████| 1.32M/1.32M [00:00<00:00, 140MB/s]

Downloaded to /content/full_dpo_dataset_llama.json


In [11]:
from datasets import load_dataset

ds = load_dataset("json", data_files="/content/full_dpo_dataset_llama.json", split="train")
ds

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['question', 'accepted_response', 'rejected_response'],
    num_rows: 400
})

In [12]:
system_prompt = """Role:
You are an expert Medical Diagnostician and Board Examiner.

Task:
Read the provided Clinical Vignette. Determine the most likely diagnosis.
You must first generate a structured clinical reasoning process inside <think> tags, followed by the final diagnosis as plain text.

Output Format:
<think>
[Explanation]
</think>

[Final Diagnosis Name]
"""

def create_dpo_messages(row):
    return {
        "prompt": [
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": row["question"]
            }
        ],
        "chosen": [
            {
                "role": "assistant",
                "content": row["accepted_response"]
            }
        ],
        "rejected": [
            {
                "role": "assistant",
                "content": row["rejected_response"]
            }
        ]
    }

ds_dpo = ds.map(
    create_dpo_messages,
    remove_columns=ds.column_names
)

ds_dpo


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 400
})

In [13]:
ds_dpo[0]

{'prompt': [{'content': 'Role:\nYou are an expert Medical Diagnostician and Board Examiner.\n\nTask:\nRead the provided Clinical Vignette. Determine the most likely diagnosis.\nYou must first generate a structured clinical reasoning process inside <think> tags, followed by the final diagnosis as plain text.\n\nOutput Format:\n<think>\n[Explanation]\n</think>\n\n[Final Diagnosis Name]\n',
   'role': 'system'},
  {'content': 'A 50-year-old male physician is medically evacuated to a high-level isolation unit after developing a febrile illness while working in Sierra Leone. His initial clinical course is dominated by severe gastrointestinal symptoms requiring aggressive fluid resuscitation. However, days after admission, his condition deteriorates significantly with the onset of profound hypoxia and respiratory failure, necessitating advanced airway management and mechanical ventilation. Chest imaging reveals diffuse pulmonary infiltrates. Diagnostic bronchoscopy is performed, and molecula

In [14]:
# This split the dataset by getting 1 row of each disease and put in the validation set
train_indices = []
val_indices = []

rows_per_disease = 5

for i in range(0, len(ds_dpo), rows_per_disease):
    chunk = list(range(i, i + rows_per_disease))

    # 1 sample → validation
    val_indices.append(chunk[0])

    # remaining samples → training
    train_indices.extend(chunk[1:])

train_dataset = ds_dpo.select(train_indices)
eval_dataset  = ds_dpo.select(val_indices)

train_dataset = train_dataset.shuffle(seed=42)

print(f"Training samples: {len(train_dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

Training samples: 320
Evaluation samples: 80


In [15]:
from peft import LoraConfig

peft_config = LoraConfig(
      r=16,
      lora_alpha=16,
      # lora_dropout=0.05,
      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

In [16]:
from trl import DPOConfig

training_args = DPOConfig(
    output_dir=output_dir,
    beta=0.1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=1e-6,
    num_train_epochs=1,
    save_steps=100,
    logging_steps=1,
    bf16=True,
    fp16=False,
    report_to="wandb",
    model_adapter_name="policy",
    ref_adapter_name="ref",
    max_length=2048,
    # use_liger_kernel=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    # Evaluation settings
    eval_strategy="steps",          # Run evaluation every eval_steps
    eval_steps=40,                  # Evaluate every 50 steps
    # eval_strategy="epoch",        # Alternative: evaluate at end of each epoch
    push_to_hub=True,
)

In [18]:
from trl import DPOTrainer

trainer = DPOTrainer(
    model=model,
    ref_model=None,           # Using adapter-based reference via ref_adapter_name
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    # peft_config=peft_config,  # Don't pass this when using existing adapters
    processing_class=tokenizer,
)

Extracting prompt in train dataset:   0%|          | 0/320 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/320 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/320 [00:00<?, ? examples/s]

Extracting prompt in eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/80 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Step,Training Loss,Validation Loss,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected,Logits/chosen,Logits/rejected
40,0.386200,0.381709,0.393698,-0.376582,1.000000,0.770280,-438.887451,-185.746048,-0.549856,-0.474555
80,0.260900,0.232839,0.648788,-0.702274,1.000000,1.351062,-436.336517,-189.002960,-0.544825,-0.461467
120,0.158800,0.172118,0.792459,-0.899464,1.000000,1.691923,-434.899811,-190.974854,-0.541852,-0.454542


In [ ]:
# Save the trained adapter locally
trainer.save_model(output_dir)

# Save the PEFT adapter separately
model.save_pretrained(f"{output_dir}")

# Push to Hugging Face Hub using trainer (recommended - handles model + tokenizer)
trainer.push_to_hub(f"{output_dir}", token=hf_token)

print(f"Model saved to {output_dir} and {output_dir}")
print(f"Pushed to Hugging Face Hub: JellyFush/{output_dir}")

In [6]:
model_id, output_dir = "meta-llama/Llama-3.2-3B-Instruct", "Llama-3.2-3B-Instruct-DPO"

In [7]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",                   # Change to Flash Attention if GPU has support
    dtype=torch.float16,                          # Change to bfloat16 if GPU has support
    device_map="auto",
    # use_cache=True,                               # Whether to cache attention outputs to speed up inference
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,                        # Load the model in 4-bit precision to save memory
        bnb_4bit_compute_dtype=torch.float16,     # Data type used for internal computations in quantization
        bnb_4bit_use_double_quant=True,           # Use double quantization to improve accuracy
        bnb_4bit_quant_type="nf4"                 # Type of quantization. "nf4" is recommended for recent LLMs
    )
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

2026-01-07 15:49:26.380789: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767800966.579430      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767800966.635149      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767800967.100776      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767800967.100821      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767800967.100824      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [10]:
messages = [
  {
      'content': """Role:
You are an expert Medical Diagnostician and Board Examiner.

Task:
Read the provided Clinical Vignette. Determine the most likely diagnosis.
You must first generate a structured clinical reasoning process inside <think> tags, followed by the final diagnosis as plain text.

Output Format:
<think>
[Explanation]
</think>

[Final Diagnosis Name]
""",
      'role': 'system',
  },
  {
      'content': """A 6-year-old boy, known to be HIV-positive but not yet on antiretroviral therapy (ART), presents with acute onset of high fever, dyspnoea, dry cough, and impaired consciousness for 3 days. He has coryza, bilateral conjunctivitis, and a fine maculopapular rash behind both ears and on the forehead that later covers the trunk with skin scaling over the chest. He also has bilateral axillary lymphadenopathy and laboured breathing. Malaria smears were negative. He is from Malawi.
      """,
      'role': 'user',
  }
]

In [9]:
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

generated_ids = base_model.generate(
    **model_inputs,
    max_new_tokens=512
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<think>
To determine the most likely diagnosis for this patient, we need to consider his clinical presentation and the context of his HIV status.

First, we should consider the differential diagnoses for a child with acute onset of high fever, dyspnea, and impaired consciousness. These symptoms can be indicative of several conditions, including infections such as pneumonia, sepsis, or meningitis.

Given the child's HIV status and the presence of a fine maculopapular rash, we should consider opportunistic infections that are more common in immunocompromised patients, such as Pneumocystis jirovecii pneumonia (PCP), cytomegalovirus (CMV) pneumonia, or tuberculosis (TB).

The presence of coryza, bilateral conjunctivitis, and axillary lymphadenopathy also raises suspicion for an infectious etiology.

However, the specific pattern of rash and skin scaling over the chest is more suggestive of measles, particularly given the child's HIV status and the fact that he is from Malawi, where measles

In [10]:
adapter_model = f"JellyFush/Llama-3.2-3B-Instruct" # Replace with your HF username or organization
fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    adapter_model
)

adapter_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/48.7M [00:00<?, ?B/s]

In [13]:
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model.device)

generated_ids = fine_tuned_model.generate(
    **model_inputs,
    max_new_tokens=2048
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<think>
1. Epidemiology/Risk: The child is HIV-positive and from Malawi, a region with high HIV prevalence. The clinical presentation of severe pneumonia with systemic symptoms like fever and impaired consciousness is highly suggestive of Pneumocystis jirovecii pneumonia (PCP), a common opportunistic infection in HIV/AIDS patients, particularly those without CD4 cell counts < 200 cells/μL.
2. Differential Exclusion: The child's clinical presentation of respiratory failure, as evidenced by impaired consciousness, bilateral lung infiltrates, and respiratory distress, is consistent with PCP, which is a medical emergency in HIV/AIDS patients. Other differential diagnoses like bacterial pneumonia are less likely given the child's HIV status and the specific constellation of symptoms.
3. Confirmation: The clinical diagnosis is confirmed by the presence of PCP on chest radiograph and microbiological confirmation by India ink staining of sputum or bronchoalveolar lavage (BAL) fluid.
</think>



In [7]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from peft import PeftModel

base_model_2 = AutoModelForCausalLM.from_pretrained(
    model_id,
    attn_implementation="sdpa",                   # Change to Flash Attention if GPU has support
    dtype=torch.float16,                          # Change to bfloat16 if GPU has support
    device_map="auto",
    # use_cache=True,                               # Whether to cache attention outputs to speed up inference
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,                        # Load the model in 4-bit precision to save memory
        bnb_4bit_compute_dtype=torch.float16,     # Data type used for internal computations in quantization
        bnb_4bit_use_double_quant=True,           # Use double quantization to improve accuracy
        bnb_4bit_quant_type="nf4"                 # Type of quantization. "nf4" is recommended for recent LLMs
    )
)

tokenizer = AutoTokenizer.from_pretrained(model_id)

2026-01-07 15:59:13.341098: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767801553.541110      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767801553.601292      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767801554.086910      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767801554.086944      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767801554.086946      55 computation_placer.cc:177] computation placer alr

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [8]:
adapter_model = f"JellyFush/Llama-3.2-3B-Instruct-DPO" # Replace with your HF username or organization
fine_tuned_model_dpo = PeftModel.from_pretrained(
    base_model_2,
    adapter_model,
    subfolder="policy", 
    assign=True
)

adapter_config.json: 0.00B [00:00, ?B/s]

policy/adapter_model.safetensors:   0%|          | 0.00/97.3M [00:00<?, ?B/s]

In [12]:
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(fine_tuned_model_dpo.device)

generated_ids = fine_tuned_model_dpo.generate(
    **model_inputs,
    max_new_tokens=2048
)
output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]

# Decode and extract model response
generated_text = tokenizer.decode(output_ids, skip_special_tokens=True)
print(generated_text)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


<think>
1. Epidemiology/Risk: The boy is HIV-positive, which places him at high risk for opportunistic infections. 2. Clinical Anchor: The combination of acute onset of respiratory symptoms (dyspnoea, cough), systemic illness (fever, impaired consciousness), and specific dermatological features (rash, scaling) is suggestive of a classic presentation. 3. Differential Exclusion: Malaria is ruled out by negative smears, and the rash pattern is not typical for malaria. The patient's immunocompromised status is a key clue, as certain infections are more likely in HIV-positive individuals. 4. Pathophysiology: The clinical picture is consistent with Pneumocystis jirovecii pneumonia (PCP), a common opportunistic infection in HIV/AIDS patients. The rash, known as 'Pneumocystis dermatitis', is a distinctive feature of PCP. 5. Confirmation: A chest X-ray showing diffuse bilateral infiltrates confirms the diagnosis.
</think>

PCP
